# MAUVE on QASA result JSONs (GPU Colab)

**MAUVE** (`mauve-text`) compares distributions of model text vs references (GPT-2–large features). It is **not** ROUGE.

1. Runtime → **Change runtime type** → **GPU** (e.g. T4).
2. Get **`eval.py`** and **`utils.py`** on Colab (clone the repo *or* upload those two files from your machine into the same folder — that folder is your `REPO`).
3. **Upload your result JSONs manually** when the file picker runs (saved under **`/content/qasa_mauve_uploads/`** — visible in the left **Files** panel). You do **not** need a `result/` directory.
4. Run the cells below in order (install → upload → batch MAUVE + merged scores under `/content/mauve_scores_colab.json`).

In [ ]:
# Minimal deps for MAUVE-only eval (no ROUGE package required)
%pip install -q mauve-text torch transformers tqdm nltk numpy

import torch
assert torch.cuda.is_available(), "Select GPU runtime: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# --- 2a) Optional: clone full repo (skip if you already uploaded eval.py + utils.py into REPO) ---
# !git clone https://github.com/<your>/<nlp_citations>.git /content/nlp_citations

from pathlib import Path

# Folder that contains eval.py and utils.py (clone repo here, OR upload those two files into this path)
REPO = Path("/content/nlp_citations")
assert (REPO / "eval.py").is_file() and (REPO / "utils.py").is_file(), (
    f"Missing eval.py or utils.py under {REPO}. Clone the repo or upload both files into REPO."
)

# --- 2b) Upload your QASA result JSON(s) from your laptop; saved under /content (visible in Files sidebar) ---
from google.colab import files

UPLOAD_DIR = Path("/content/qasa_mauve_uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
print("Choose your result JSON file(s)…")
for name, data in files.upload().items():
    if not name.endswith(".json"):
        print(f"Skipping non-JSON: {name}")
        continue
    dest = UPLOAD_DIR / name
    dest.write_bytes(data)
    print("Saved:", dest)

json_paths = sorted(UPLOAD_DIR.glob("*.json"))
assert json_paths, f"No .json files in {UPLOAD_DIR}"
print("Ready:", [str(p) for p in json_paths])
print("Next: run the batch cell to compute MAUVE for all of these (writes <name>.score next to each JSON).")

In [ ]:
# Batch MAUVE: every .json in UPLOAD_DIR (no tools/batch_mauve.py needed)
import subprocess, sys, json
from pathlib import Path

REPO = Path("/content/nlp_citations")
UPLOAD_DIR = Path("/content/qasa_mauve_uploads")
paths = sorted(UPLOAD_DIR.glob("*.json"))
assert paths, f"No JSON in {UPLOAD_DIR}"

merged = {}
for p in paths:
    cmd = [sys.executable, "eval.py", "--f", str(p), "--mauve_only", "--mauve_device", "0"]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO), check=True)
    score_path = Path(str(p) + ".score")
    merged[p.name] = json.loads(score_path.read_text())

out = Path("/content/mauve_scores_colab.json")
out.write_text(json.dumps(merged, indent=2), encoding="utf-8")
print("Wrote", out)
print(json.dumps(merged, indent=2))